In [2]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 35.2 MB/s eta 0:00:00


In [3]:
!pip install -q "trl==0.8.6" "transformers==4.40.0" "peft==0.10.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 3.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 80.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 64.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 13.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


In [3]:
import torch

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU count      : {torch.cuda.device_count()}")
print(f"GPU name       : {torch.cuda.get_device_name(0)}")
print(f"Torch version  : {torch.__version__}")

CUDA available : True
GPU count      : 2
GPU name       : Tesla T4
Torch version  : 2.10.0+cu128


In [4]:
import pandas as pd
from datasets import Dataset

# Load our pre-processed ChatML dataset
df = pd.read_csv('/kaggle/input/datasets/hagargalall/chatml-dataset-csv/chatml_dataset.csv')

print(f"Total rows : {len(df)}")
print(f"Columns    : {df.columns.tolist()}")
print("\nSample:")
print(df['text'].iloc[0][:300])

Total rows : 50329
Columns    : ['text']

Sample:
<|im_start|>system
أنت مساعد طبي ذكي ومتخصص يعمل باللغة العربية.
مهمتك هي الإجابة على الأسئلة الطبية بدقة علمية واحترافية.
تذكر دائماً: أنت مساعد للمعلومات وليس بديلاً عن الطبيب المختص.<|im_end|>
<|im_start|>user
ما هي مميزات و عيوب الدواء جلوكوفانس 500 5 و ايضا الانسولين مكس تارد 30<|im_end|>
<|im_


In [5]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Split 95/5
train_df, eval_df = train_test_split(df, test_size=0.05, random_state=42)

# Convert to HuggingFace Dataset
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
eval_ds  = Dataset.from_pandas(eval_df.reset_index(drop=True))

print(f"Train samples      : {len(train_ds)}")
print(f"Validation samples : {len(eval_ds)}")

Train samples      : 47812
Validation samples : 2517


In [6]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Tokenizer loaded  ")
print(f"Vocab size        : {tokenizer.vocab_size}")
print(f"Pad token         : {tokenizer.pad_token}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Tokenizer loaded  
Vocab size        : 151643
Pad token         : <|im_end|>


In [7]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False
print("Model loaded in 4-bit ")
print(f"Parameters: {model.num_parameters():,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model loaded in 4-bit 
Parameters: 494,032,768


In [8]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497346861940344


In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/kaggle/working/qwen-medical-lora",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=25,
    evaluation_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    optim="paged_adamw_8bit",
)

print("Training arguments set")

Training arguments set


In [12]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,
)

print("Starting training")
trainer.train()
print("Training complete!")

Map:   0%|          | 0/47812 [00:00<?, ? examples/s]

Map:   0%|          | 0/2517 [00:00<?, ? examples/s]

Starting training


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
100,1.924300,1.865506
200,1.369800,1.273243
300,1.101700,1.108167
400,1.137600,1.045963
500,1.093800,1.009661
600,0.948000,0.979547
700,1.023000,0.968069
800,0.957300,0.948934
900,0.957900,0.938871
1000,0.973400,0.931636


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a n

Training complete!


In [4]:
import os

path = "/kaggle/working/qwen-medical-lora"

if os.path.exists(path):
    print("Files found:")
    for f in os.listdir(path):
        print(f"  {f}")
else:
    print(" No saved files found — need to retrain")

Files found:
  checkpoint-5800
  checkpoint-5600


In [5]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_ID    = "Qwen/Qwen2.5-0.5B-Instruct"
CHECKPOINT  = "/kaggle/working/qwen-medical-lora/checkpoint-5800"
MERGED_DIR  = "/kaggle/working/qwen-medical-merged"

print("Loading base model for merging...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True,
)

print("Loading LoRA adapter from checkpoint...")
model = PeftModel.from_pretrained(base_model, CHECKPOINT)

print("Merging weights...")
model = model.merge_and_unload()

print("Saving merged model...")
model.save_pretrained(MERGED_DIR, safe_serialization=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.save_pretrained(MERGED_DIR)

print("Merged model saved to:", MERGED_DIR)

Loading base model for merging...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Merged model saved to: /kaggle/working/qwen-medical-merged


In [6]:
import shutil

print("Zipping merged model...")
shutil.make_archive(
    '/kaggle/working/qwen-medical-merged',
    'zip',
    '/kaggle/working/qwen-medical-merged'
)
print("Zip created!")

# Check zip size
import os
size = os.path.getsize('/kaggle/working/qwen-medical-merged.zip')
print(f"Zip size: {size / (1024*1024*1024):.2f} GB")

Zipping merged model...
Zip created!
Zip size: 0.81 GB


In [7]:
import os
os.makedirs('/kaggle/working/output', exist_ok=True)

shutil.copy(
    '/kaggle/working/qwen-medical-merged.zip',
    '/kaggle/working/output/qwen-medical-merged.zip'
)
print("Done")

Done


In [11]:
import os

# Create a dataset output folder
os.makedirs('/kaggle/working/dataset_output', exist_ok=True)

# Copy zip there
import shutil
shutil.copy(
    '/kaggle/working/qwen-medical-merged.zip',
    '/kaggle/working/dataset_output/qwen-medical-merged.zip'
)
print("Done")

Done
